# Optimization

## Learning Objectives
1. Implement SGD, Adam, Adagrad, and RMSProp from scratch in NumPy.
2. Compare all optimisers on a neural network in PyTorch with OOM handling.
3. Apply AdamW for LLM fine-tuning with weight decay and gradient clipping.
4. Visualise the 2D loss landscape and optimiser trajectories.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: SGD / Adam / Adagrad / RMSProp from Scratch (NumPy)


In [ ]:
def make_quadratic(n=300, d=5):
    """Simple quadratic regression task."""
    X = np.random.randn(n, d).astype(np.float32)
    w_true = np.array([1.0, -2.0, 0.5, 1.5, -1.0])
    y = X @ w_true + 0.1 * np.random.randn(n)
    return X, y.astype(np.float32), w_true


def mse_grad(w, X, y):
    """Gradient of MSE loss w.r.t. w."""
    pred = X @ w
    return (2 / len(X)) * X.T @ (pred - y)


class SGDScratch:
    def __init__(self, lr=0.01): self.lr = lr; self.w = None
    def init(self, d): self.w = np.zeros(d)
    def step(self, g): self.w -= self.lr * g


class AdamScratch:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr=lr; self.b1=beta1; self.b2=beta2; self.eps=eps
        self.m=self.v=None; self.t=0; self.w=None
    def init(self, d):
        self.w=np.zeros(d); self.m=np.zeros(d); self.v=np.zeros(d)
    def step(self, g):
        self.t+=1
        self.m=self.b1*self.m+(1-self.b1)*g
        self.v=self.b2*self.v+(1-self.b2)*g**2
        mh=self.m/(1-self.b1**self.t); vh=self.v/(1-self.b2**self.t)
        self.w-=self.lr*mh/(np.sqrt(vh)+self.eps)


class AdagradScratch:
    def __init__(self, lr=0.1, eps=1e-8): self.lr=lr; self.eps=eps; self.G=None; self.w=None
    def init(self, d): self.w=np.zeros(d); self.G=np.zeros(d)
    def step(self, g): self.G+=g**2; self.w-=self.lr*g/(np.sqrt(self.G)+self.eps)


class RMSPropScratch:
    def __init__(self, lr=0.01, alpha=0.99, eps=1e-8):
        self.lr=lr; self.alpha=alpha; self.eps=eps; self.v=None; self.w=None
    def init(self, d): self.w=np.zeros(d); self.v=np.zeros(d)
    def step(self, g):
        self.v=self.alpha*self.v+(1-self.alpha)*g**2
        self.w-=self.lr*g/(np.sqrt(self.v)+self.eps)


X_q, y_q, w_true = make_quadratic()
optimisers = {"SGD": SGDScratch(0.05), "Adam": AdamScratch(0.01),
              "Adagrad": AdagradScratch(0.1), "RMSProp": RMSPropScratch(0.01)}

print(f"{'Optimiser':>10} | {'Final MSE':>10} | {'||w - w*||':>12}")
print("-" * 38)
for name, opt in optimisers.items():
    opt.init(5)
    for _ in range(500):
        g = mse_grad(opt.w, X_q, y_q)
        opt.step(g)
    mse = np.mean((X_q @ opt.w - y_q) ** 2)
    dist = np.linalg.norm(opt.w - w_true)
    print(f"{name:>10} | {mse:>10.6f} | {dist:>12.6f}")


## Level 2: Optimiser Comparison on MLP (PyTorch + OOM Handling)


In [ ]:
class OptMLP(nn.Module):
    def __init__(self, in_dim=20, hidden=128, out_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim))
    def forward(self, x): return self.net(x)


X_o = torch.randn(2000, 20); y_o = torch.randint(0, 5, (2000,))
opt_loader = DataLoader(TensorDataset(X_o[:1600], y_o[:1600]),
                        batch_size=64, shuffle=True)
X_ov, y_ov = X_o[1600:].to(device), y_o[1600:].to(device)


def run_optimizer(opt_cls, opt_kwargs, n_epochs=50):
    m = OptMLP().to(device)
    opt = opt_cls(m.parameters(), **opt_kwargs)
    crit = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(n_epochs):
        m.train()
        for xb, yb in opt_loader:
            opt.zero_grad()
            try:
                loss = crit(m(xb.to(device)), yb.to(device))
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print("OOM: reduce batch_size"); torch.cuda.empty_cache(); continue
                raise
            loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            val_accs.append((m(X_ov).argmax(1)==y_ov).float().mean().item())
    return val_accs


opt_configs = {
    "SGD":          (torch.optim.SGD,    {"lr": 0.05}),
    "SGD+Momentum": (torch.optim.SGD,    {"lr": 0.05, "momentum": 0.9}),
    "Adagrad":      (torch.optim.Adagrad,{"lr": 0.01}),
    "RMSProp":      (torch.optim.RMSprop,{"lr": 0.01}),
    "Adam":         (torch.optim.Adam,   {"lr": 1e-3}),
    "AdamW":        (torch.optim.AdamW,  {"lr": 1e-3, "weight_decay": 1e-2}),
}
opt_results = {}
for name, (cls, kwargs) in opt_configs.items():
    opt_results[name] = run_optimizer(cls, kwargs)
    print(f"{name:>14}: final val acc = {opt_results[name][-1]:.4f}")


## Real-World Example 1: AdamW for LLM Fine-Tuning Style Training


In [ ]:
class TinyTransformerLM(nn.Module):
    """Minimal transformer for AdamW fine-tuning demo."""
    def __init__(self, vocab_size=500, d_model=128, nhead=4,
                 n_layers=2, n_classes=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        return self.classifier(self.encoder(self.embedding(x))[:, 0, :])


model_lm = TinyTransformerLM().to(device)

no_decay = {'bias', 'LayerNorm.weight'}
param_groups = [
    {"params": [p for n, p in model_lm.named_parameters()
                if not any(nd in n for nd in no_decay)],
     "weight_decay": 0.01},
    {"params": [p for n, p in model_lm.named_parameters()
                if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
opt_lm = torch.optim.AdamW(param_groups, lr=5e-4, betas=(0.9, 0.999))
total_steps = 200
scheduler_lm = torch.optim.lr_scheduler.CosineAnnealingLR(opt_lm, T_max=total_steps)

crit_lm = nn.CrossEntropyLoss()
X_lm = torch.randint(0, 500, (1000, 32)); y_lm = torch.randint(0, 5, (1000,))
lm_loader = DataLoader(TensorDataset(X_lm, y_lm), batch_size=32, shuffle=True)

lm_losses = []
for step, (xb, yb) in enumerate(lm_loader):
    if step >= total_steps: break
    opt_lm.zero_grad()
    loss_lm = crit_lm(model_lm(xb.to(device)), yb.to(device))
    loss_lm.backward()
    nn.utils.clip_grad_norm_(model_lm.parameters(), 1.0)
    opt_lm.step(); scheduler_lm.step()
    lm_losses.append(loss_lm.item())
    if step % 50 == 0:
        print(f"Step {step:3d} | loss: {loss_lm.item():.4f} | "
              f"lr: {opt_lm.param_groups[0]['lr']:.2e}")


## Real-World Example 2: Gradient Clipping for RNN Stability


In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=64, output_size=5):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


def train_rnn(clip_value=None, n_epochs=30, lr=0.01):
    """Train RNN with or without gradient clipping; return loss history."""
    m = VanillaRNN().to(device)
    o = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    c = nn.CrossEntropyLoss()
    X_r = torch.randn(500, 20, 10); y_r = torch.randint(0, 5, (500,))
    rnn_loader = DataLoader(TensorDataset(X_r, y_r), batch_size=32, shuffle=True)
    hist = []
    for epoch in range(n_epochs):
        total = 0.0
        for xb, yb in rnn_loader:
            o.zero_grad()
            loss = c(m(xb.to(device)), yb.to(device))
            loss.backward()
            if clip_value:
                nn.utils.clip_grad_norm_(m.parameters(), clip_value)
            o.step()
            total += loss.item()
        avg = total / len(rnn_loader)
        hist.append(avg)
        if np.isnan(avg) or avg > 1e6:
            print(f"Diverged at epoch {epoch+1}"); break
    return hist


hist_clip   = train_rnn(clip_value=1.0)
hist_noclip = train_rnn(clip_value=None)

print(f"With    clipping: final loss = {hist_clip[-1]:.4f}")
print(f"Without clipping: final loss = {hist_noclip[-1]:.4f}")


## Real-World Example 3: 2D Loss Landscape Visualisation


In [ ]:
def rosenbrock_2d(x0, x1, a=1.0, b=100.0):
    return (a - x0)**2 + b*(x1 - x0**2)**2


xx = np.linspace(-2.5, 2.5, 300)
yy = np.linspace(-0.5, 3.0, 300)
XX, YY = np.meshgrid(xx, yy)
ZZ = rosenbrock_2d(XX, YY)


def run_2d_optimiser(name, lr=0.001, n_steps=1000):
    """Run numpy optimiser on 2D Rosenbrock and return path."""
    x = np.array([-1.5, 1.5])
    v = np.zeros(2)
    m2, v2 = np.zeros(2), np.zeros(2)
    eps_adam = 1e-8; b1, b2 = 0.9, 0.999
    path = [x.copy()]
    for t in range(1, n_steps + 1):
        g = np.array([
            -2*(1-x[0]) - 4*100*x[0]*(x[1]-x[0]**2),
            2*100*(x[1]-x[0]**2)
        ])
        if name == "sgd":
            x = x - lr * g
        elif name == "momentum":
            v = 0.9*v - lr*g; x = x + v
        elif name == "adam":
            m2=b1*m2+(1-b1)*g; v2=b2*v2+(1-b2)*g**2
            mh=m2/(1-b1**t); vh=v2/(1-b2**t)
            x=x-lr*mh/(np.sqrt(vh)+eps_adam)
        path.append(x.copy())
    return np.array(path)


paths_2d = {
    "SGD":      run_2d_optimiser("sgd",      lr=0.0008),
    "Momentum": run_2d_optimiser("momentum", lr=0.0008),
    "Adam":     run_2d_optimiser("adam",     lr=0.005),
}

fig, ax = plt.subplots(figsize=(9, 5))
ax.contourf(XX, YY, np.log1p(ZZ), levels=30, cmap='Blues')
ax.contour(XX, YY, np.log1p(ZZ), levels=30, colors='white', alpha=0.3, linewidths=0.5)
colors_2d = {"SGD": "tomato", "Momentum": "gold", "Adam": "limegreen"}
for name, path in paths_2d.items():
    ax.plot(path[:, 0], path[:, 1], '-', color=colors_2d[name],
            lw=1.5, label=name, alpha=0.85)
    ax.plot(path[0, 0], path[0, 1], 'o', color=colors_2d[name], ms=6)
    ax.plot(path[-1, 0], path[-1, 1], '*', color=colors_2d[name], ms=10)
ax.plot(1, 1, 'w*', ms=14, label='Minimum (1,1)')
ax.set_xlabel('x0'); ax.set_ylabel('x1')
ax.set_title('Rosenbrock Loss Landscape + Optimiser Paths')
ax.legend(loc='upper left')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-0.5, 3.0)
plt.tight_layout()
plt.savefig('/tmp/optimization_landscape.png', dpi=80)
plt.close()
print('Loss landscape saved to /tmp/optimization_landscape.png')
